# Research Paper Analysis and Recommendation System

Assignment 3 use case: build an NLP/LLM system that analyzes arXiv papers, extracts metadata, summarizes findings, identifies research gaps, and recommends relevant papers.

Implemented techniques:

- Basic NLP: preprocessing, TF-IDF keyword extraction, NER/rule-based metadata extraction, traditional text classification, embedding similarity.
- Advanced NLP/LLM: RAG, LoRA fine-tuning setup, instruction prompting, privacy/fairness/ethics-aware LLM prompting, and LLM summarization/research-gap generation.

In [2]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
MAX_PAPERS = 8000
DATA_FILE = Path('arxiv-metadata-oai-snapshot.json')

## 1. Load arXiv Metadata

Place `arxiv-metadata-oai-snapshot.json` in this folder. If it is missing, the cell can download it with `kagglehub`.

In [3]:
def resolve_dataset_path() -> Path:
    if DATA_FILE.exists():
        return DATA_FILE

    try:
        import kagglehub
    except ImportError as exc:
        raise FileNotFoundError(
            'Download the Kaggle arXiv dataset and place '
            'arxiv-metadata-oai-snapshot.json in this folder, '
            'or install kagglehub.'
        ) from exc

    dataset_dir = Path(kagglehub.dataset_download('Cornell-University/arxiv'))
    return dataset_dir / 'arxiv-metadata-oai-snapshot.json'


dataset_path = resolve_dataset_path()
dataset_path

PosixPath('arxiv-metadata-oai-snapshot.json')

In [4]:
def load_arxiv_subset(path: Path, max_rows: int = MAX_PAPERS) -> pd.DataFrame:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            categories = item.get('categories', '')
            # Keep the demo focused on CS/AI/NLP-style papers.
            if any(
                cat.startswith(('cs.', 'stat.ML'))
                for cat in categories.split()
            ):
                rows.append(item)
            if len(rows) >= max_rows:
                break

    df = pd.DataFrame(rows)
    useful_cols = [
        'id',
        'title',
        'authors',
        'abstract',
        'categories',
        'comments',
        'journal-ref',
        'doi',
        'update_date',
    ]
    df = df[[col for col in useful_cols if col in df.columns]].dropna(
        subset=['title', 'abstract', 'categories']
    )
    df['title'] = df['title'].str.replace(r'\s+', ' ', regex=True).str.strip()
    df['abstract'] = (
        df['abstract'].str.replace(r'\s+', ' ', regex=True).str.strip()
    )
    df['paper_text'] = df['title'] + '. ' + df['abstract']
    df['primary_category'] = df['categories'].str.split().str[0]
    return df.reset_index(drop=True)


papers = load_arxiv_subset(dataset_path)
papers.head()

,id,title,authors,abstract,categories,comments,journal-ref,doi,update_date,paper_text,primary_category
0,0704.0002,Sparsity-certifying Graph Decompositions,Ileana Streinu and Louis Theran,"We describe a new algorithm, the $(k,\ell)$-pe...",math.CO cs.CG,To appear in Graphs and Combinatorics,NaN,NaN,2008-12-13,Sparsity-certifying Graph Decompositions. We d...,math.CO
1,0704.0046,A limit relation for entropy and channel capac...,"I. Csiszar, F. Hiai and D. Petz","In a quantum mechanical model, Diosi, Feldmann...",quant-ph cs.IT math.IT,"LATEX file, 11 pages","J. Math. Phys. 48(2007), 092102.",10.1063/1.2779138,2009-11-13,A limit relation for entropy and channel capac...,quant-ph
2,0704.0047,Intelligent location of simultaneously active ...,T. Kosel and I. Grabec,The intelligent acoustic emission locator is d...,cs.NE cs.AI,"5 pages, 5 eps figures, uses IEEEtran.cls",NaN,NaN,2009-09-29,Intelligent location of simultaneously active ...,cs.NE
3,0704.0050,Intelligent location of simultaneously active ...,T. Kosel and I. Grabec,Part I describes an intelligent acoustic emiss...,cs.NE cs.AI,"5 pages, 7 eps figures, uses IEEEtran.cls",NaN,NaN,2007-05-23,Intelligent location of simultaneously active ...,cs.NE
4,0704.0062,On-line Viterbi Algorithm and Its Relationship...,"Rastislav \v{S}r\'amek, Bro\v{n}a Brejov\'a, T...","In this paper, we introduce the on-line Viterb...",cs.DS,NaN,Algorithms in Bioinformatics: 7th Internationa...,10.1007/978-3-540-74126-8_23,2010-01-25,On-line Viterbi Algorithm and Its Relationship...,cs.DS


## 2. Basic Techniques




### 2.1 Text Preprocessing

In [5]:
STOPWORDS = {
    'a',
    'an',
    'and',
    'are',
    'as',
    'at',
    'be',
    'by',
    'for',
    'from',
    'has',
    'in',
    'is',
    'it',
    'its',
    'of',
    'on',
    'or',
    'that',
    'the',
    'this',
    'to',
    'we',
    'with',
    'using',
    'used',
    'via',
}


def preprocess(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s-]', ' ', text)
    tokens = [
        token
        for token in text.split()
        if len(token) > 2 and token not in STOPWORDS
    ]
    return ' '.join(tokens)


papers['clean_text'] = papers['paper_text'].map(preprocess)
papers[['title', 'clean_text']].head()

,title,clean_text
0,Sparsity-certifying Graph Decompositions,sparsity-certifying graph decompositions descr...
1,A limit relation for entropy and channel capac...,limit relation entropy channel capacity per un...
2,Intelligent location of simultaneously active ...,intelligent location simultaneously active aco...
3,Intelligent location of simultaneously active ...,intelligent location simultaneously active aco...
4,On-line Viterbi Algorithm and Its Relationship...,on-line viterbi algorithm relationship random ...


### 2.2 TF-IDF Keyword Extraction

TF-IDF gives interpretable paper-level keywords for metadata extraction and recommendation explanation.

In [6]:
tfidf = TfidfVectorizer(
    max_features=12000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.75,
)
tfidf_matrix = tfidf.fit_transform(papers['clean_text'])
terms = np.array(tfidf.get_feature_names_out())


def top_keywords(row_index: int, k: int = 8) -> list[str]:
    row = tfidf_matrix[row_index].toarray().ravel()
    if row.sum() == 0:
        return []
    top_idx = row.argsort()[-k:][::-1]
    return terms[top_idx].tolist()


papers['keywords'] = [top_keywords(i) for i in range(len(papers))]
papers[['title', 'keywords']].head()

,title,keywords
0,Sparsity-certifying Graph Decompositions,"[ell, sparse graphs, sparse, game, colors, dec..."
1,A limit relation for entropy and channel capac...,"[conjecture, unit cost, per unit, entropy, qua..."
2,Intelligent location of simultaneously active ...,"[locator, emission, acoustic, intelligent, loc..."
3,Intelligent location of simultaneously active ...,"[emission, acoustic, sources, separation, ica,..."
4,On-line Viterbi Algorithm and Its Relationship...,"[viterbi algorithm, viterbi, on line, hmms, on..."


### 2.3 Metadata Extraction / NER

arXiv metadata already contains authors. The rule-based extractor below also detects likely institutions in abstracts/comments when available. If `en_core_web_sm` is installed, spaCy NER can be added for richer organization extraction.

In [7]:
INSTITUTION_PATTERN = re.compile(
    r'\b(?:University|Institute|Laboratory|Lab|College|School|Centre|Center)'
    r'\s+of\s+[A-Z][A-Za-z\-]+|'
    r'\b[A-Z][A-Za-z\-]+\s+'
    r'(?:University|Institute|Laboratory|Lab|College|School|Centre|Center)\b'
)


def parse_authors(authors: str) -> list[str]:
    return [
        name.strip()
        for name in re.split(r',| and ', authors or '')
        if name.strip()
    ]


def extract_institutions(text: str) -> list[str]:
    return sorted(
        set(
            match.group(0)
            for match in INSTITUTION_PATTERN.finditer(text or '')
        )
    )


papers['author_list'] = papers['authors'].map(parse_authors)
comments = (
    papers['comments'].fillna('')
    if 'comments' in papers.columns
    else pd.Series('', index=papers.index)
)
papers['institutions'] = (comments + ' ' + papers['abstract']).map(
    extract_institutions
)
papers[['title', 'author_list', 'institutions']].head()

,title,author_list,institutions
0,Sparsity-certifying Graph Decompositions,"[Ileana Streinu, Louis Theran]",[]
1,A limit relation for entropy and channel capac...,"[I. Csiszar, F. Hiai, D. Petz]",[]
2,Intelligent location of simultaneously active ...,"[T. Kosel, I. Grabec]",[]
3,Intelligent location of simultaneously active ...,"[T. Kosel, I. Grabec]",[]
4,On-line Viterbi Algorithm and Its Relationship...,"[Rastislav \v{S}r\'amek, Bro\v{n}a Brejov\'a, ...",[]


### 2.4 Traditional Text Classification

A TF-IDF + Logistic Regression baseline predicts the primary arXiv domain. This gives a traditional NLP comparison point against embedding/LLM methods.

In [55]:
common_categories = papers['primary_category'].value_counts().head(8).index
clf_data = papers[papers['primary_category'].isin(common_categories)].copy()

X_train, X_test, y_train, y_test = train_test_split(
    clf_data['clean_text'],
    clf_data['primary_category'],
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=clf_data['primary_category'],
)

classifier = Pipeline(
    [
        (
            'tfidf',
            TfidfVectorizer(
                max_features=20000,
                ngram_range=(1, 2),
                min_df=2,
            ),
        ),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ]
)
classifier.fit(X_train, y_train)
pred = classifier.predict(X_test)
print(classification_report(y_test, pred, zero_division=0))

              precision    recall  f1-score   support

       cs.AI       0.72      0.70      0.71        88
       cs.CC       0.58      0.59      0.58        64
       cs.DM       0.61      0.67      0.64        81
       cs.DS       0.67      0.75      0.71       110
       cs.IT       0.97      0.84      0.90       433
       cs.LO       0.77      0.84      0.80        97
       cs.NI       0.65      0.84      0.73        95
       cs.OH       0.86      0.86      0.86        84

    accuracy                           0.79      1052
   macro avg       0.73      0.76      0.74      1052
weighted avg       0.81      0.79      0.80      1052



### 2.5 Dense Embedding Search

Sentence embeddings support semantic paper recommendation and provide the retriever for RAG.

In [56]:
from sentence_transformers import SentenceTransformer

embedding_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(embedding_model_name)

embeddings = embedding_model.encode(
    papers['paper_text'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

retriever = NearestNeighbors(n_neighbors=8, metric='cosine')
retriever.fit(embeddings)
embeddings.shape

Batches: 100%|██████████| 125/125 [00:11<00:00, 10.76it/s]


(8000, 384)

In [57]:
def recommend_papers(query: str, top_k: int = 5) -> pd.DataFrame:
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )
    distances, indices = retriever.kneighbors(
        query_embedding,
        n_neighbors=top_k,
    )
    results = papers.iloc[indices[0]].copy()
    results['similarity'] = 1 - distances[0]
    return results[
        [
            'id',
            'title',
            'authors',
            'categories',
            'keywords',
            'similarity',
            'abstract',
        ]
    ]


query = (
    'retrieval augmented generation for scientific literature review and '
    'citation recommendation'
)
recommendations = recommend_papers(query, top_k=5)
recommendations[['title', 'categories', 'keywords', 'similarity']]

,title,categories,keywords,similarity
129,Using Access Data for Paper Recommendations on...,cs.DL cs.IR,"[citation, access, org, recommendation, arxiv,...",0.589187
5147,Peer-review in the Internet age,physics.soc-ph cs.DL,"[review, peer, growing, scientific, internet, ...",0.574038
5717,An evaluation of Bradfordizing effects,cs.DL,"[articles, document, core, zones, zone, inform...",0.538805
4015,Universality of citation distributions: toward...,physics.soc-ph cond-mat.stat-mech cs.DL physic...,"[indicator, disciplines, citation, distributio...",0.526644
3085,"Citation Counting, Citation Ranking, and h-Ind...",cs.HC cs.IR,"[citation, web science, science, web, hci, res...",0.513336


## 3. Advanced Techniques



### 3.1 RAG for Paper Search and Related Work Discovery

RAG combines the embedding retriever with a grounded context window so the LLM can recommend papers and discuss related work using retrieved evidence.

In [58]:
def build_rag_context(
    results: pd.DataFrame,
    max_chars_per_paper: int = 900,
) -> str:
    blocks = []
    for _, row in results.iterrows():
        abstract = row['abstract'][:max_chars_per_paper]
        blocks.append(
            f"Paper ID: {row['id']}\n"
            f"Title: {row['title']}\n"
            f"Authors: {row['authors']}\n"
            f"Categories: {row['categories']}\n"
            f"Keywords: {', '.join(row['keywords'])}\n"
            f"Abstract: {abstract}"
        )
    return '\n\n---\n\n'.join(blocks)


rag_context = build_rag_context(recommendations)
print(rag_context[:2500])

Paper ID: 0704.2963
Title: Using Access Data for Paper Recommendations on ArXiv.org
Authors: Stefan Pohl
Categories: cs.DL cs.IR
Keywords: citation, access, org, recommendation, arxiv, data, access data, data source
Abstract: This thesis investigates in the use of access log data as a source of information for identifying related scientific papers. This is done for arXiv.org, the authority for publication of e-prints in several fields of physics. Compared to citation information, access logs have the advantage of being immediately available, without manual or automatic extraction of the citation graph. Because of that, a main focus is on the question, how far user behavior can serve as a replacement for explicit meta-data, which potentially might be expensive or completely unavailable. Therefore, we compare access, content, and citation-based measures of relatedness on different recommendation tasks. As a final result, an online recommendation system has been built that can help scient

### 3.2 Fine-Tuning Setup Using LoRA

LoRA is a parameter-efficient fine-tuning method. This setup shows how the recommender could adapt an instruction model to arXiv-style recommendation responses without updating all model weights. The training call is optional because full fine-tuning can be slow on CPU.

In [59]:
def build_lora_training_preview(sample_papers: pd.DataFrame) -> list[dict[str, str]]:
    examples = []
    for _, row in sample_papers.head(3).iterrows():
        prompt_text = (
            'Recommend this paper for a literature review and explain its fit.\n'
            f"Title: {row['title']}\n"
            f"Categories: {row['categories']}\n"
            f"Abstract: {row['abstract'][:700]}"
        )
        response_text = (
            f"Recommendation: {row['id']} - {row['title']}\n"
            f"Reason: This paper is relevant to {row['categories']} and matches the retrieved topic."
        )
        examples.append({'prompt': prompt_text, 'response': response_text})
    return examples


lora_training_preview = build_lora_training_preview(recommendations)
lora_training_preview[0]


{'prompt': 'Recommend this paper for a literature review and explain its fit.\nTitle: Using Access Data for Paper Recommendations on ArXiv.org\nCategories: cs.DL cs.IR\nAbstract: This thesis investigates in the use of access log data as a source of information for identifying related scientific papers. This is done for arXiv.org, the authority for publication of e-prints in several fields of physics. Compared to citation information, access logs have the advantage of being immediately available, without manual or automatic extraction of the citation graph. Because of that, a main focus is on the question, how far user behavior can serve as a replacement for explicit meta-data, which potentially might be expensive or completely unavailable. Therefore, we compare access, content, and citation-based measures of relatedness on different recommendation tasks. As a final r',
 'response': 'Recommendation: 0704.2963 - Using Access Data for Paper Recommendations on ArXiv.org\nReason: This paper

In [60]:
def show_lora_config() -> dict[str, object]:
    try:
        from peft import LoraConfig, TaskType

        config = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=['q', 'v'],
        )
        return config.to_dict()
    except Exception as exc:
        return {
            'status': 'LoRA config preview only',
            'reason': str(exc),
            'install': 'pip install peft',
        }


show_lora_config()


{'task_type': <TaskType.SEQ_2_SEQ_LM: 'SEQ_2_SEQ_LM'>,
 'peft_type': <PeftType.LORA: 'LORA'>,
 'auto_mapping': None,
 'base_model_name_or_path': None,
 'revision': None,
 'inference_mode': False,
 'r': 8,
 'target_modules': {'q', 'v'},
 'exclude_modules': None,
 'lora_alpha': 16,
 'lora_dropout': 0.05,
 'fan_in_fan_out': False,
 'bias': 'none',
 'use_rslora': False,
 'modules_to_save': None,
 'init_lora_weights': True,
 'layers_to_transform': None,
 'layers_pattern': None,
 'rank_pattern': {},
 'alpha_pattern': {},
 'megatron_config': None,
 'megatron_core': 'megatron.core',
 'trainable_token_indices': None,
 'loftq_config': {},
 'eva_config': None,
 'corda_config': None,
 'use_dora': False,
 'use_qalora': False,
 'qalora_group_size': 16,
 'layer_replication': None,
 'lora_bias': False}

### 3.3 Prompt Engineering

The prompt uses explicit instructions, structured output requirements, and grounding constraints.

In [61]:
def make_rag_prompt(user_question: str, context: str) -> str:
    return f"""
You are an academic research assistant.
Use only the retrieved paper context below.

Task:
1. Summarize the main research direction in 4 bullet points.
2. Recommend the 3 most relevant papers and explain why.
3. Identify 2 likely research gaps.
4. Suggest one future research question.

Grounding rules:
- Cite paper IDs from the context.
- If the context is insufficient, say what is missing.
- Do not invent papers, authors, or results.

User question: {user_question}

Retrieved context:
{context}

Answer:
""".strip()


prompt = make_rag_prompt(query, rag_context)
print(prompt[:3000])

You are an academic research assistant.
Use only the retrieved paper context below.

Task:
1. Summarize the main research direction in 4 bullet points.
2. Recommend the 3 most relevant papers and explain why.
3. Identify 2 likely research gaps.
4. Suggest one future research question.

Grounding rules:
- Cite paper IDs from the context.
- If the context is insufficient, say what is missing.
- Do not invent papers, authors, or results.

User question: retrieval augmented generation for scientific literature review and citation recommendation

Retrieved context:
Paper ID: 0704.2963
Title: Using Access Data for Paper Recommendations on ArXiv.org
Authors: Stefan Pohl
Categories: cs.DL cs.IR
Keywords: citation, access, org, recommendation, arxiv, data, access data, data source
Abstract: This thesis investigates in the use of access log data as a source of information for identifying related scientific papers. This is done for arXiv.org, the authority for publication of e-prints in several f